In [1]:
import torch
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms

import os
import numpy as np

class Load_Dataset(Dataset):
    # Initialize your data, download, etc.
    def __init__(self, dataset):
        super(Load_Dataset, self).__init__()

        # Load samples
        x_data = dataset["samples"]

        # Convert to torch tensor
        if isinstance(x_data, np.ndarray):
            x_data = torch.from_numpy(x_data)

        # Load labels
        y_data = dataset.get("labels")
        if y_data is not None and isinstance(y_data, np.ndarray):
            y_data = torch.from_numpy(y_data)

        self.x_data = x_data.float()
        self.y_data = y_data.long() if y_data is not None else None

        self.len = x_data.shape[0]

    def get_labels(self):
        return self.y_data

    def __getitem__(self, index):
        sample = {
            'samples': self.x_data[index].squeeze(-1),
            'labels': int(self.y_data[index])
        }

        return sample

    def __len__(self):
        return self.len


In [16]:
def data_generator(data_path, batch_size = 64):
    # original
    train_dataset = torch.load(data_path)
    val_dataset = torch.load(data_path)
    test_dataset = torch.load(data_path)

    # Loading datasets
    train_dataset = Load_Dataset(train_dataset)
    val_dataset = Load_Dataset(val_dataset)
    test_dataset = Load_Dataset(test_dataset)

    cw = train_dataset.y_data.numpy().tolist()
    cw_dict = {}
    for i in range(len(np.unique(train_dataset.y_data.numpy()))):
        cw_dict[i] = cw.count(i)
    # print(cw_dict)

    # Dataloaders
    train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size,
                                               shuffle=True, drop_last=True, num_workers=0)
    val_loader = torch.utils.data.DataLoader(dataset=val_dataset, batch_size=batch_size,
                                             shuffle=False, drop_last=True, num_workers=0)
    test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size,
                                              shuffle=False, drop_last=False, num_workers=0)
    return train_loader, val_loader, test_loader



In [33]:
import pandas as pd

train_dl, val_dl, test_dl = data_generator("../data/mit/train.pt", batch_size=64)
x_np = train_dl.dataset.x_data.detach().numpy()  # Convert PyTorch tensor to NumPy array
# Nếu x là 2D (N, d) -> DataFrame trực tiếp
if x_np.ndim == 2:
    df = pd.DataFrame(x_np)

# Nếu x là 3D (N, T, d) -> flatten thành (N, T*d) để thành DataFrame
elif x_np.ndim == 3:
    N, T, D = x_np.shape
    df = pd.DataFrame(x_np.reshape(N, T * D))
    # hoặc nếu bạn muốn MultiIndex cột:
    # df = pd.DataFrame(x_np.reshape(N, T*D),
    #                   columns=pd.MultiIndex.from_product([range(T), range(D)], names=["t", "feat"]))

# Nếu nhiều chiều hơn -> flatten theo chiều đầu (N, -1)
else:
    df = pd.DataFrame(x_np.reshape(x_np.shape[0], -1))

print(df.iloc[0, :-1])


0      1.000000
1      0.431250
2      0.446875
3      0.418750
4      0.400000
         ...   
181    0.000000
182    0.000000
183    0.000000
184    0.000000
185    0.000000
Name: 0, Length: 186, dtype: float32


In [ ]:
Load_Dataset